# Modelo Recomendador

El modelo recomendador se basa en la similitud de audio features mediante similitud coseno. Cuanto más cercanos son dos vectores de features, más parecidas son las canciones.

## 1. Importación de librerías y carga del dataset



In [1]:
# Librerías de manipulación de datos.
import pandas as pd 
import numpy as np 

# Librerías de machine learning.
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

# Configuración de pandas.
pd.set_option('display.max_columns', None)

In [2]:
# Carga del dataset.
df_cluster = pd.read_csv('../data/processed/tracks_clustered.csv')
df = df_cluster.copy()
print(f'Dataset cargado: {df.shape[0]:,} filas')
df.head(3)

Dataset cargado: 113,422 filas


,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,duration_min,popularity_category,cluster,cluster_nombre
0,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.461,1,-6.746,0,0.1430,0.0322,0.000001,0.358,0.715,87.917,4,acoustic,3.84,Alta,3,Fiesta & Baile
1,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.166,1,-17.235,1,0.0763,0.9240,0.000006,0.101,0.267,77.489,4,acoustic,2.49,Alta,1,Acústico & Tranquilo
2,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.359,0,-9.734,1,0.0557,0.2100,0.000000,0.117,0.120,76.332,4,acoustic,3.51,Alta,1,Acústico & Tranquilo


## 2. Matriz de variables

In [3]:
# Selección de variables.
variables_modelo = ['danceability', 'energy', 'loudness', 'speechiness','acousticness', 'instrumentalness', 'liveness','valence', 'tempo']

# Escala de las variables.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[variables_modelo])

print(f'Matriz de variables: {X_scaled.shape}')
print(f'Variables usadas : {variables_modelo}')


Matriz de variables: (113422, 9)
Variables usadas : ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']


## 3. Función de recomendación

In [17]:
def recomendar_canciones(titulo, artista=None, n=10, df=df, X_scaled=X_scaled):
    """ 
    Recomienda canciones similares a sus variables dado el título de una canción.
    Args:
        titulo: Nombre de la canción (búsqueda parcial)
        artista: Filtrar por artista (opcional)
        n:  Número de recomendaciones
    
    Returns:
        DataFrame con las n canciones más similares
    """
    
    # Buscar la canción en el dataset.
    mask = df['track_name'].str.contains(titulo, case=False, na=False)
    if artista:
        mask &= df['artists'].str.contains(artista, case=False, na=False)
    candidatos = df[mask]
    
    if candidatos.empty:
        print(f'No se encontraron canciones que contengan "{titulo}"')
        return None
    
    # Si hay varias coincidencias, tomar la más popular.
    idx = candidatos['popularity'].idxmax()
    cancion_ref = df.loc[idx]
    
    print(f'Canción de referencia:')
    print(f"   {cancion_ref['track_name']} — {cancion_ref['artists']}")
    print(f'    Género: {cancion_ref["track_genre"]} | Cluster: {cancion_ref["cluster_nombre"]}')
    print(f'    Popularidad: {cancion_ref["popularity"]}')
    
    # Calcular similitud coseno entre la canción y todas las demás
    vector_ref = X_scaled[idx].reshape(1, -1)
    similitudes = cosine_similarity(vector_ref, X_scaled)[0]
    
    # Ordenar por similitud descendente, excluir la propia canción
    indices_similares = np.argsort(similitudes)[::-1]
    indices_similares = [i for i in indices_similares if i != idx][:n]
    
    recomendaciones = df.iloc[indices_similares][
        ['track_name', 'artists', 'track_genre', 'cluster_nombre', 'popularity',
         'danceability', 'energy', 'valence']
    ].copy()
    recomendaciones['similitud'] = similitudes[indices_similares].round(4)
    recomendaciones = recomendaciones.reset_index(drop=True)
    recomendaciones.index += 1
    
    return recomendaciones



In [18]:
# Primera prueba
titulo = "Blinding Lights"
artista = "The Weeknd"
n = 10
recomendaciones = recomendar_canciones(titulo, artista, n)
recomendaciones

Canción de referencia:
   Blinding Lights — The Weeknd
    Género: pop | Cluster: Rock & Intenso
    Popularidad: 91


,track_name,artists,track_genre,cluster_nombre,popularity,danceability,energy,valence,similitud
1,平凡人的自傳 - Rap Version,ONE PROMISE,cantopop,Rock & Intenso,22,0.507,0.717,0.353,0.9916
2,Viah,Jass Manak,hip-hop,Rock & Intenso,59,0.533,0.757,0.358,0.9914
3,Thinkin About,ShockOne;Lee Mvtthews,j-dance,Rock & Intenso,52,0.555,0.681,0.337,0.9881
4,Thinkin About,ShockOne;Lee Mvtthews,drum-and-bass,Rock & Intenso,52,0.555,0.681,0.337,0.9881
5,Fool Yourself,Chase & Status;Plan B;Rage,drum-and-bass,Rock & Intenso,22,0.471,0.781,0.255,0.9872
6,30/90,Andrew Garfield;Joshua Henry;Vanessa Hudgens;R...,show-tunes,Rock & Intenso,67,0.466,0.789,0.360,0.9867
7,BODY,LICK;LUNA AURA,club,Rock & Intenso,43,0.488,0.674,0.370,0.9861
8,Danger Line,Avenged Sevenfold,metal,Rock & Intenso,58,0.473,0.767,0.375,0.9854
9,Broken,Netsky;Montell2099,drum-and-bass,Rock & Intenso,54,0.518,0.806,0.327,0.9849
10,Blinding Lights,The Weeknd,pop,Rock & Intenso,3,0.512,0.796,0.344,0.9820


Comprobamos que muestran valores muy similares entre entre los parámetros, y que pertenecen todos al mismo cluster. 


In [19]:
# Segunda prueba.
titulo = "Shape of You"
artista = "Ed Sheeran"
n = 10
recomendaciones = recomendar_canciones(titulo, artista, n)
recomendaciones

Canción de referencia:
   Shape of You — Ed Sheeran
    Género: pop | Cluster: Fiesta & Baile
    Popularidad: 86


,track_name,artists,track_genre,cluster_nombre,popularity,danceability,energy,valence,similitud
1,Tene,Larry Gaaga;Flavour,dancehall,Fiesta & Baile,0,0.830,0.766,0.932,0.9768
2,Yaaro,Santesh;Amos Paul,malay,Fiesta & Baile,29,0.801,0.673,0.789,0.9766
3,Brujeria,El Gran Combo De Puerto Rico,salsa,Fiesta & Baile,65,0.795,0.623,0.960,0.9766
4,Go-Go Club - Raw,Vybz Kartel,j-dance,Fiesta & Baile,20,0.786,0.767,0.863,0.9758
5,Pop the Bubbles,Patty Shukla,kids,Fiesta & Baile,33,0.820,0.721,0.904,0.9749
6,Separemos Nuestras Vidas,Jerry Rivera,salsa,Fiesta & Baile,30,0.803,0.702,0.894,0.9739
7,Quiero Llenarte,Jerry Rivera,salsa,Fiesta & Baile,35,0.814,0.716,0.841,0.9723
8,Low,Larry Gaaga;Wizkid,dancehall,Fiesta & Baile,54,0.762,0.587,0.772,0.9716
9,Pull Up,Timaya;Burna Boy,dancehall,Fiesta & Baile,53,0.847,0.700,0.880,0.9706
10,Whine It,Jahyanai;Timal,dancehall,Fiesta & Baile,36,0.723,0.646,0.799,0.9701
